# 03 - Correlação e Associação

## Objetivo

Avaliar estatisticamente as relações identificadas durante a análise exploratória
e identificar quais variáveis apresentam associação relevante com a renovação
das apólices.

Serão utilizadas diferentes técnicas de acordo com o tipo de variável:

- Qui-quadrado: associação entre variáveis categóricas;
- Cramér's V: força da associação entre variáveis categóricas;
- Correlação de Spearman: relação entre variáveis numéricas e a variável alvo;
- Information Value (IV): poder preditivo das variáveis em relação à renovação.

### Pergunta de negócio

> Quais características apresentam evidências estatísticas de associação com a
> renovação das apólices?

## 3.2. Imports

In [2]:
import pandas as pd
import numpy as np

from scipy.stats import chi2_contingency
from scipy.stats import spearmanr

pd.set_option("display.max_columns", None)

## 3.3. Carregamento de dados

In [3]:
df = pd.read_excel(
    "../data/raw/base-seguros.xlsx",
    sheet_name="Base"
)

## 3.3. Separando variáveis

In [4]:
variaveis_categoricas = [
    "Perfil_Risco",
    "Diferenca_Perfil",
    "Genero",
    "Profissao",
    "Uso_Veiculo",
    "Premio_Qte_Parc",
    "Veic_Garagem",
    "Veic_Potencia",
    "Veic_Regiao"
]

In [5]:
variaveis_numericas = [
    "Idade",
    "Tempo_Apolice",
    "Qte_Apolices",
    "Premio_Final",
    "Premio_Pago_Ult",
    "Premio_Mercado",
    "Premio_Orig",
    "Veic_Idade",
    "Veic_Idade_Compra"
]

## 3.4. Criar função para Qui-quadrado + Cramér's V

In [6]:
def qui_quadrado_cramers_v(df, variavel, target="Flag_Renovou"):

    tabela = pd.crosstab(
        df[variavel],
        df[target]
    )

    chi2, p_value, dof, expected = chi2_contingency(tabela)

    n = tabela.sum().sum()
    r, k = tabela.shape

    cramers_v = np.sqrt(
        chi2 / (n * min(k - 1, r - 1))
    )

    return {
        "variavel": variavel,
        "chi2": chi2,
        "p_value": p_value,
        "cramers_v": cramers_v
    }

## 3.5. Aplicar a todas as categóricas

In [7]:
resultados_categoricas = []

for variavel in variaveis_categoricas:

    resultado = qui_quadrado_cramers_v(
        df,
        variavel
    )

    resultados_categoricas.append(resultado)

resultados_categoricas = pd.DataFrame(
    resultados_categoricas
)

resultados_categoricas.sort_values(
    "cramers_v",
    ascending=False
)

,variavel,chi2,p_value,cramers_v
0,Perfil_Risco,179.213396,1.214243e-39,0.088157
8,Veic_Regiao,90.676967,1.037137e-13,0.062707
4,Uso_Veiculo,73.275809,1.225633e-16,0.056370
5,Premio_Qte_Parc,38.007775,2.815782e-08,0.040598
1,Diferenca_Perfil,35.124706,4.076257e-06,0.039028
6,Veic_Garagem,21.508646,3.085934e-03,0.030541
2,Genero,8.414137,3.723144e-03,0.019102
3,Profissao,7.891522,4.966704e-03,0.018499
7,Veic_Potencia,7.176842,7.086512e-01,0.017642


## 3.6. Criar uma classificação da força da associação

In [11]:
def classificar_cramers_v(v):

    if v < 0.10:
        return "Muito fraca"

    elif v < 0.20:
        return "Fraca"

    elif v < 0.30:
        return "Moderada"

    elif v < 0.50:
        return "Forte"

    else:
        return "Muito forte"

In [12]:
resultados_categoricas["forca_associacao"] = (
    resultados_categoricas["cramers_v"]
    .apply(classificar_cramers_v)
)

In [13]:
resultados_categoricas = resultados_categoricas.sort_values(
    "cramers_v",
    ascending=False
)

resultados_categoricas

,variavel,chi2,p_value,cramers_v,forca_associacao
0,Perfil_Risco,179.213396,1.214243e-39,0.088157,Muito fraca
8,Veic_Regiao,90.676967,1.037137e-13,0.062707,Muito fraca
4,Uso_Veiculo,73.275809,1.225633e-16,0.056370,Muito fraca
5,Premio_Qte_Parc,38.007775,2.815782e-08,0.040598,Muito fraca
1,Diferenca_Perfil,35.124706,4.076257e-06,0.039028,Muito fraca
6,Veic_Garagem,21.508646,3.085934e-03,0.030541,Muito fraca
2,Genero,8.414137,3.723144e-03,0.019102,Muito fraca
3,Profissao,7.891522,4.966704e-03,0.018499,Muito fraca
7,Veic_Potencia,7.176842,7.086512e-01,0.017642,Muito fraca


## 3.7. Aplicar a todas as numéricas

In [14]:
resultados_numericas = []

for variavel in variaveis_numericas:

    correlacao, p_value = spearmanr(
        df[variavel],
        df["Flag_Renovou"]
    )

    resultados_numericas.append({
        "variavel": variavel,
        "spearman": correlacao,
        "p_value": p_value
    })

resultados_numericas = pd.DataFrame(
    resultados_numericas
)

In [15]:
resultados_numericas["abs_spearman"] = (
    resultados_numericas["spearman"].abs()
)

resultados_numericas.sort_values(
    "abs_spearman",
    ascending=False
)

,variavel,spearman,p_value,abs_spearman
0,Idade,-0.061576,8.053120e-21,0.061576
7,Veic_Idade,-0.051552,4.755173e-15,0.051552
4,Premio_Pago_Ult,0.050819,1.148113e-14,0.050819
3,Premio_Final,0.047535,5.120423e-13,0.047535
5,Premio_Mercado,0.047081,8.485382e-13,0.047081
1,Tempo_Apolice,-0.045464,4.947579e-12,0.045464
6,Premio_Orig,0.040894,5.223343e-10,0.040894
8,Veic_Idade_Compra,-0.010534,1.096736e-01,0.010534
2,Qte_Apolices,-0.004648,4.802950e-01,0.004648


## 3.8. Information Value

### 3.8.1. Criar bins

In [16]:
def criar_bins(df, variavel, q=10):

    return pd.qcut(
        df[variavel],
        q=q,
        duplicates="drop"
    )

### 3.8.2. Calcular WoE e IV

In [17]:
def calcular_iv(df, variavel, target="Flag_Renovou"):

    temp = df[[variavel, target]].copy()

    if pd.api.types.is_numeric_dtype(temp[variavel]):

        temp["bin"] = pd.qcut(
            temp[variavel],
            q=10,
            duplicates="drop"
        )

    else:

        temp["bin"] = temp[variavel].astype(str)

    tabela = pd.crosstab(
        temp["bin"],
        temp[target]
    )

    if 0 not in tabela.columns:
        tabela[0] = 0

    if 1 not in tabela.columns:
        tabela[1] = 0

    tabela = tabela[[0, 1]]

    tabela["dist_0"] = tabela[0] / tabela[0].sum()
    tabela["dist_1"] = tabela[1] / tabela[1].sum()

    epsilon = 1e-10

    tabela["woe"] = np.log(
        (tabela["dist_1"] + epsilon) /
        (tabela["dist_0"] + epsilon)
    )

    tabela["iv"] = (
        tabela["dist_1"] - tabela["dist_0"]
    ) * tabela["woe"]

    iv = tabela["iv"].sum()

    return iv

### 3.8.3. Calcular IV para todas as variáveis

In [19]:
variaveis_iv = (
    variaveis_categoricas +
    variaveis_numericas
)

In [20]:
resultados_iv = []

for variavel in variaveis_iv:

    iv = calcular_iv(
        df,
        variavel
    )

    resultados_iv.append({
        "variavel": variavel,
        "iv": iv
    })

resultados_iv = pd.DataFrame(
    resultados_iv
)

In [23]:
resultados_iv.sort_values(
    "iv",
    ascending=False
)

,variavel,iv
0,Perfil_Risco,6.904373e-02
10,Tempo_Apolice,4.939806e-02
9,Idade,4.246317e-02
8,Veic_Regiao,3.519826e-02
4,Uso_Veiculo,3.303143e-02
16,Veic_Idade,2.673983e-02
13,Premio_Pago_Ult,2.583342e-02
14,Premio_Mercado,2.561783e-02
12,Premio_Final,2.198651e-02
17,Veic_Idade_Compra,1.845373e-02


### 3.8.4. Classificar o IV

In [24]:
def classificar_iv(iv):

    if iv < 0.02:
        return "Muito fraco"

    elif iv < 0.10:
        return "Fraco"

    elif iv < 0.30:
        return "Moderado"

    elif iv < 0.50:
        return "Forte"

    else:
        return "Muito forte"

In [25]:
resultados_iv["forca"] = (
    resultados_iv["iv"]
    .apply(classificar_iv)
)

resultados_iv.sort_values(
    "iv",
    ascending=False
)

,variavel,iv,forca
0,Perfil_Risco,6.904373e-02,Fraco
10,Tempo_Apolice,4.939806e-02,Fraco
9,Idade,4.246317e-02,Fraco
8,Veic_Regiao,3.519826e-02,Fraco
4,Uso_Veiculo,3.303143e-02,Fraco
16,Veic_Idade,2.673983e-02,Fraco
13,Premio_Pago_Ult,2.583342e-02,Fraco
14,Premio_Mercado,2.561783e-02,Fraco
12,Premio_Final,2.198651e-02,Fraco
17,Veic_Idade_Compra,1.845373e-02,Muito fraco


### 3.8.5. Ranking final

In [26]:
ranking = resultados_iv.copy()

ranking = ranking.sort_values(
    "iv",
    ascending=False
)

ranking

,variavel,iv,forca
0,Perfil_Risco,6.904373e-02,Fraco
10,Tempo_Apolice,4.939806e-02,Fraco
9,Idade,4.246317e-02,Fraco
8,Veic_Regiao,3.519826e-02,Fraco
4,Uso_Veiculo,3.303143e-02,Fraco
16,Veic_Idade,2.673983e-02,Fraco
13,Premio_Pago_Ult,2.583342e-02,Fraco
14,Premio_Mercado,2.561783e-02,Fraco
12,Premio_Final,2.198651e-02,Fraco
17,Veic_Idade_Compra,1.845373e-02,Muito fraco


In [27]:
df["Nao_Renovou"] = (df["Flag_Renovou"] == 0).astype(int)

In [28]:
principais_variaveis = [
    "Perfil_Risco",
    "Veic_Regiao",
    "Uso_Veiculo",
    "Premio_Qte_Parc",
    "Diferenca_Perfil"
]

for variavel in principais_variaveis:

    resultado = (
        df.groupby(variavel)["Nao_Renovou"]
          .mean()
          .mul(100)
          .sort_values(ascending=False)
          .rename("taxa_nao_renovacao")
    )

    print("=" * 70)
    print(variavel)
    display(resultado)

Perfil_Risco


Perfil_Risco
up        90.218642
stable    89.772350
down      83.870015
Name: taxa_nao_renovacao, dtype: float64

Veic_Regiao


Veic_Regiao
Reg2     92.625899
Reg5     89.962359
Reg1     89.482201
Reg7     88.149556
Reg6     87.935657
Reg3     87.794198
Reg4     87.329480
Reg8     87.020169
Reg10    86.849574
Reg13    85.570470
Reg9     85.029940
Reg11    84.587814
Reg14    84.173778
Reg12    81.962264
Name: taxa_nao_renovacao, dtype: float64

Uso_Veiculo


Uso_Veiculo
unknown                      91.645134
private or freelance work    86.400572
commercial                   80.000000
Name: taxa_nao_renovacao, dtype: float64

Premio_Qte_Parc


Premio_Qte_Parc
4 per year     89.025188
12 per year    88.648897
2 per year     87.119741
1 per year     85.976027
Name: taxa_nao_renovacao, dtype: float64

Diferenca_Perfil


Diferenca_Perfil
commercial          95.000000
all drivers > 24    88.136574
same                87.745406
only partner        87.167815
unknown             83.333333
young drivers       83.273657
learner 17          80.952381
Name: taxa_nao_renovacao, dtype: float64

## Conclusão

A análise estatística mostrou que diversas variáveis apresentam associação
estatisticamente significativa com a renovação.

Entretanto, as magnitudes observadas são predominantemente baixas.

`Perfil_Risco` apresentou a maior associação entre as variáveis categóricas
(Cramér's V = 0,088), enquanto `Idade` apresentou a maior correlação monotônica
entre as variáveis numéricas (ρ = -0,062).

O Information Value também indicou que nenhuma variável isoladamente possui
forte poder de discriminação. O maior IV foi observado para `Perfil_Risco`
(0,069).

Esses resultados sugerem que o comportamento de renovação provavelmente não é
determinado por um único fator, mas pela combinação de múltiplas características.

As análises realizadas nesta etapa serão utilizadas como suporte à construção
do modelo preditivo, sem eliminar automaticamente variáveis apenas com base em
uma única métrica estatística.

### Principais hipóteses para a modelagem

- Perfil de risco;
- Tempo de apólice;
- Idade do cliente;
- Região;
- Uso do veículo;
- Características relacionadas ao prêmio.